In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import json
import pandas as pd
import random
import datetime
from tqdm import tqdm
from utils import geo_to_mercator, mercator_to_geo

import cv2


In [2]:
random.seed(123)
np.random.seed(123)

In [3]:
geo_prior_fp = "geo_prior_train.csv"
geo_prior_data = pd.read_csv(geo_prior_fp)
geo_prior_data = geo_prior_data.drop('quality_grade', axis=1)   # drop quality_grade since all are "research"
geo_prior_data = geo_prior_data.drop('positional_accuracy', axis=1)   # drop positional_accuracy since not really needed (and most are NaN)
geo_prior_data.shape

(35500262, 6)

Notes: Landsat was launched Feb 11, 2013, Sentinel-2A launched June 23, 2015, and Sentinel-2B launched March 7, 2017 (see https://hls.gsfc.nasa.gov/wp-content/uploads/2019/01/HLS.v1.4.UserGuide_draft_ver3.1.pdf)

NAIP data is from 2011-2020

Planet data is from 2016

In [4]:
geo_prior_data['observed_on'] = pd.to_datetime(geo_prior_data['observed_on'], format='%Y-%m-%d') 
geo_prior_data['year'] = pd.to_datetime(geo_prior_data['observed_on']).dt.strftime('%Y').astype(int)

geo_prior_data = geo_prior_data[geo_prior_data["year"]>=2017]     # only get from 2017 since we only have Sentinel-2B data starting 2017

In [5]:
geo_prior_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 31520093 entries, 57496 to 35500261
Data columns (total 7 columns):
 #   Column            Dtype         
---  ------            -----         
 0   observation_uuid  object        
 1   observer_id       int64         
 2   latitude          float64       
 3   longitude         float64       
 4   taxon_id          int64         
 5   observed_on       datetime64[ns]
 6   year              int64         
dtypes: datetime64[ns](1), float64(2), int64(3), object(1)
memory usage: 1.9+ GB


# Sample max number of samples per species

In [6]:
max_num_per_species = 1000  # max number of samples to use per class

# samp = geo_prior_data.groupby("taxon_id").sample(n=max_num_per_species, random_state=123)
df = geo_prior_data.iloc[np.random.permutation(len(geo_prior_data))]    # shuffle dataframe
samp = df.groupby("taxon_id").head(max_num_per_species)
samp.shape

(13731767, 7)

In [7]:
samp["taxon_id"].value_counts()

taxon_id
18236      1000
52856      1000
904334     1000
127457     1000
54412      1000
           ... 
102102        2
1193458       2
1283181       2
521039        1
446326        1
Name: count, Length: 47373, dtype: int64

In [8]:

samp['month'] = pd.to_datetime(samp['observed_on']).dt.strftime('%m').astype(int)
samp['day'] = pd.to_datetime(samp['observed_on']).dt.strftime('%d').astype(int)

/tmp/ipykernel_394518/3377052998.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  samp['month'] = pd.to_datetime(samp['observed_on']).dt.strftime('%m').astype(int)
/tmp/ipykernel_394518/3377052998.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  samp['day'] = pd.to_datetime(samp['observed_on']).dt.strftime('%d').astype(int)


In [9]:
samp.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13731767 entries, 2392749 to 24503293
Data columns (total 9 columns):
 #   Column            Dtype         
---  ------            -----         
 0   observation_uuid  object        
 1   observer_id       int64         
 2   latitude          float64       
 3   longitude         float64       
 4   taxon_id          int64         
 5   observed_on       datetime64[ns]
 6   year              int64         
 7   month             int64         
 8   day               int64         
dtypes: datetime64[ns](1), float64(2), int64(5), object(1)
memory usage: 1.0+ GB


In [10]:
samp.to_csv("sampled_geo_prior_data.csv", index=False)

# Match SINR and Satlas Sentinel Data

In [ ]:
sentinel2_dir = "/datasets/ai/allenai/satlas_pretrain/sentinel2"

In [4]:

all_files = []
for dir_name in tqdm(os.listdir(sentinel2_dir)):
    datetime_str = dir_name.split("_")[2]
    date_obj = datetime.datetime.strptime(datetime_str, "%Y%m%dT%H%M%S")
    if not os.path.exists(os.path.join(sentinel2_dir, dir_name, "tci")):
        print(f"{dir_name} doesn't have tci folder")
        continue
    for filename in  os.listdir(os.path.join(sentinel2_dir, dir_name, "tci")):
        # print(filename)
        fp = os.path.join(dir_name, "tci", filename)
        col, row = filename.split(".")[0].split("_")
        all_files.append([fp, datetime_str, date_obj.year, date_obj.month, date_obj.day, col, row])
        # break
    # break

  9%|▉         | 18830/204129 [18:21<3:00:40, 17.09it/s] 


KeyboardInterrupt: 

In [58]:
df = pd.DataFrame(all_files, columns=["fp","datetime", "year", "month", "day", "col", "row"])
df.to_csv("sentinel2_files.csv", index=False)
df

,fp,datetime,year,month,day,col,row
0,S2A_MSIL1C_20220105T114501_N0301_R123_T30UVG_2...,20220105T114501,2022,1,5,3992,2557
1,S2A_MSIL1C_20220105T114501_N0301_R123_T30UVG_2...,20220105T114501,2022,1,5,3992,2558
2,S2A_MSIL1C_20220105T114501_N0301_R123_T30UVG_2...,20220105T114501,2022,1,5,3992,2559
3,S2A_MSIL1C_20220105T114501_N0301_R123_T30UVG_2...,20220105T114501,2022,1,5,3992,2561
4,S2A_MSIL1C_20220105T114501_N0301_R123_T30UVG_2...,20220105T114501,2022,1,5,3992,2562
...,...,...,...,...,...,...,...
5849942,S2B_MSIL1C_20221230T180749_N0509_R041_T13UET_2...,20221230T180749,2022,12,30,1736,2717
5849943,S2B_MSIL1C_20221230T180749_N0509_R041_T13UET_2...,20221230T180749,2022,12,30,1738,2711
5849944,S2B_MSIL1C_20221230T180749_N0509_R041_T13UET_2...,20221230T180749,2022,12,30,1740,2699
5849945,S2B_MSIL1C_20221230T180749_N0509_R041_T13UET_2...,20221230T180749,2022,12,30,1740,2700


In [ ]:

def f(lon, lat):
    col,row = geo_to_mercator((lon,lat), zoom=13, pixels=1)
    return (col,row)
sinr = pd.read_csv("sampled_geo_prior_data.csv")
sinr['col-row'] = sinr.apply(lambda x: f(x['longitude'], x['latitude']), axis=1)
sinr["col"] = sinr.apply(lambda x: x["col-row"][0], axis=1)
sinr["row"] = sinr.apply(lambda x: x["col-row"][1], axis=1)
sinr.to_csv("sinr_with_mercator.csv", index=False)

In [ ]:
sinr = pd.read_csv("sinr_with_mercator.csv")

In [61]:
sinr["col"] = (sinr["col"]).astype(int)
sinr["row"] = (sinr["row"]).astype(int)
df["col"] = (df["col"]).astype(int)
df["row"] = (df["row"]).astype(int)

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4662012 entries, 0 to 4662011
Data columns (total 7 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   fp        object
 1   datetime  object
 2   year      int64 
 3   month     int64 
 4   day       int64 
 5   col       int64 
 6   row       int64 
dtypes: int64(5), object(2)
memory usage: 249.0+ MB
